# 🐧 PenG — Intelligent Study Assistant (Google Colab T4 GPU Edition)
Notebook tối ưu riêng cho **Thuyết trình & Live Demo tốc độ cao (0 giây trễ)**.

> ⚠️ **Yêu cầu trước khi chạy:**
> Vào menu **Runtime** ➔ **Change runtime type** ➔ Chọn **T4 GPU** ➔ Bấm **Save**.

In [ ]:
# 1. Kiểm tra môi trường phần cứng GPU T4
!nvidia-smi
import torch
assert torch.cuda.is_available(), '❌ LỖI: Chưa kích hoạt GPU! Vào Runtime -> Change runtime type -> T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Đã kết nối GPU: {gpu_name} ({vram_gb:.1f} GB VRAM) sẵn sàng!')

In [ ]:
# 2. Clone hoặc cập nhật mã nguồn PenG từ nhánh main
import os
if os.path.basename(os.getcwd()) != 'PenG':
    if not os.path.exists('PenG'):
        !git clone https://github.com/canhcutlo/PenG.git
    %cd PenG
!git checkout main
!git pull origin main

In [ ]:
# 3. Cài đặt các thư viện hệ thống và Python cho Colab GPU
print('⏳ Đang cài đặt gói hệ thống OCR và FFmpeg...')
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie tesseract-ocr-eng ffmpeg > /dev/null 2>&1

print('⏳ Đang cài đặt thư viện Python (khoảng 1-2 phút)...')
!pip install -r requirements-colab.txt -q
!pip install pyngrok nest-asyncio -q

print('✅ Cài đặt môi trường hoàn tất!')

In [ ]:
# 4. Cấu hình GPU & Tải trước mô hình (Pre-warm Cache) - BÍ QUYẾT DEMO TỨC THÌ
import os

# Thiết lập biến môi trường chạy GPU với Transformers 4-bit (NF4)
os.environ['LLM_MODEL'] = 'Qwen/Qwen2.5-1.5B-Instruct'
os.environ['LLM_DEVICE'] = 'cuda'
os.environ['LLM_QUANTIZE'] = 'true'
os.environ['LLM_RUNTIME'] = 'transformers'
os.environ['AUTH_COOKIE_SECURE'] = 'true'

print('🚀 Đang nạp trước các mô hình AI vào bộ nhớ GPU (chỉ cần chạy 1 lần)...')
from app.services.llm import complete, embed, _get_embedding_model, _get_llm

print('1/2 Nạp & Khởi động Embedding Model (Vietnamese-SBERT)... ')
_get_embedding_model()
await embed(['Khởi động embedding'])

print('2/2 Nạp & Khởi động LLM Qwen2.5-1.5B trên GPU T4 (4-bit VRAM ~1.5GB)... ')
_get_llm()
await complete('Xin chào, sẵn sàng demo.')

print('\n🎉 TẤT CẢ MÔ HÌNH ĐÃ SẴN SÀNG TRONG VRAM! Mọi thao tác demo sẽ phản hồi trong 1-2 giây!')

In [ ]:
# 5. Khởi chạy Server PenG & Mở đường dẫn Public URL (Cloudflare Tunnel - Không cần Token)
import subprocess, time, os, requests, re

# Dọn dẹp tiến trình cũ nếu chạy lại cell
os.system('fuser -k 8000/tcp 2>/dev/null')
os.system('pkill -f cloudflared 2>/dev/null')
time.sleep(1)

# Khởi chạy uvicorn server ngầm, chuyển hướng log ra file server.log
log_file = open('server.log', 'w')
server_proc = subprocess.Popen(
    ['uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print('⏳ Đang khởi động server PenG...')
server_ready = False
for _ in range(30):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8000/api/health', timeout=1)
        if r.status_code == 200:
            server_ready = True
            print('✅ Server PenG đã sẵn sàng trên cổng 8000!')
            break
    except Exception:
        pass

if not server_ready:
    print('❌ Lỗi khởi động server. Đang kiểm tra log:')
    !tail -n 20 server.log
else:
    # Cài đặt Cloudflared (Tạo HTTPS tunnel hoàn toàn miễn phí, không giới hạn session)
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

    cf_log = open('cloudflared.log', 'w')
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=cf_log,
        stderr=subprocess.STDOUT,
    )

    tunnel_url = None
    print('⏳ Đang khởi tạo đường truyền HTTPS bảo mật qua Cloudflare...')
    for _ in range(30):
        time.sleep(1)
        if os.path.exists('cloudflared.log'):
            with open('cloudflared.log', 'r', errors='ignore') as f_cf:
                content = f_cf.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                tunnel_url = match.group(0)
                break

    if tunnel_url:
        print('\n' + '='*65)
        print('🐧 LIÊN KẾT GIAO DIỆN DEMO PENG (Click vào link dưới để mở):')
        print(f'👉 {tunnel_url}')
        print(f'\n📖 API Swagger Documentation: {tunnel_url}/docs')
        print('='*65)
        print('💡 Gợi ý: Mở liên kết trên bằng tab mới để sẵn sàng thuyết trình!')
    else:
        print('⚠️ Chưa lấy được URL Cloudflare tự động. Dưới đây là log cloudflared:')
        !tail -n 20 cloudflared.log


In [ ]:
# 6. (Tùy chọn) Chạy bằng ngrok nếu bạn có sẵn ngrok authtoken
NGROK_AUTHTOKEN = ''  # Điền token của bạn nếu muốn dùng ngrok

if NGROK_AUTHTOKEN.strip():
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    tunnel = ngrok.connect(8000)
    print('Ngrok Public URL:', tunnel.public_url)
else:
    print('ℹ️ Bỏ qua cell này nếu đã dùng Cloudflare Tunnel ở cell 5.')

In [ ]:
# 7. Pre-flight Check: Tự động kiểm tra kịch bản demo (Mất ~3 giây trên GPU)
# Bạn nên chạy cell này 1 lần trước khi lên sân khấu để đảm bảo mọi luồng hoạt động 100% trơn tru.
import httpx, time, uuid

client = httpx.Client(base_url='http://localhost:8000', timeout=60.0)
test_user = f'presenter_{uuid.uuid4().hex[:6]}'
client.post('/api/auth/register', json={'username': test_user, 'password': 'DemoPassword123!'})
client.post('/api/auth/login', json={'username': test_user, 'password': 'DemoPassword123!'})
csrf = client.cookies.get('peng_csrf', '')
headers = {'X-CSRF-Token': csrf}

print('1. Tải lên tài liệu mẫu (sample/sample.txt)... ')
with open('sample/sample.txt', 'rb') as f:
    r = client.post('/api/upload', files={'file': ('sample.txt', f, 'text/plain')}, data={'category': 'pdf'}, headers=headers)
doc_id = r.json()['doc_id']
job_id = r.json().get('job_id')

print('2. Chờ tiến trình xử lý nền (GPU)... ')
for _ in range(30):
    r_job = client.get(f'/api/jobs/{job_id}', headers=headers)
    if r_job.json().get('status') == 'completed':
        break
    time.sleep(1)

t_mm = time.time()
r_mm = client.get(f'/api/mindmap/{doc_id}', headers=headers)
print(f'3. Sơ đồ tư duy (Mindmap): {time.time()-t_mm:.2f}s ✅')

t_qz = time.time()
r_qz = client.post(f'/api/quiz/generate?doc_id={doc_id}&num_questions=2', headers=headers)
print(f'4. Câu hỏi trắc nghiệm (Quiz 2 câu): {time.time()-t_qz:.2f}s ✅')

t_ch = time.time()
r_sess = client.post('/api/chat/sessions', json={'doc_id': doc_id, 'title': 'Demo QA'}, headers=headers)
sess_id = r_sess.json()['session_id']
r_msg = client.post(f'/api/chat/{sess_id}/messages', json={'content': 'Nguyên lý hòn tuyết lăn xuất phát từ hình ảnh gì?'}, headers=headers)
print(f'5. Hỏi đáp có trích dẫn (Chat): {time.time()-t_ch:.2f}s ✅')
print(f'   Trả lời: {r_msg.json().get("answer")}')

print('\n🚀 SẴN SÀNG! Mọi tính năng hoạt động siêu tốc trên GPU T4, bạn có thể tự tin demo!')

In [ ]:
# 8. Xem log server trong trường hợp cần kiểm tra sự cố
!tail -n 30 server.log

## 📋 Hướng dẫn xử lý tình huống nhanh khi thuyết trình

| Tình huống | Cách xử lý tức thì |
|---|---|
| **Link Cloudflare bị mất kết nối** | Chạy lại **Cell 5** để nhận link mới trong 3 giây. |
| **Hết phiên Colab (Disconnected)** | Bấm **Reconnect**, chạy từ **Cell 1** đến **Cell 5** (chỉ mất ~2 phút vì model đã cache). |
| **Muốn restart server nhanh** | Chạy lại **Cell 5**. Cell này tự động ngắt cổng 8000 cũ và khởi tạo server mới. |
| **Kiểm tra lỗi backend** | Chạy **Cell 8** để xem 30 dòng log mới nhất. |